# Sesión 3 · Limpieza de datos (pipeline AutoGest)

Reproduce el pipeline del curso sobre el dataset del proyecto aplicando **RNF-03 (MAR 'No aplica')** e **IQR por tipo de servicio**.

Ver `sesiones/s3/limpieza_autogest.py`.

In [ ]:
import unicodedata
import pandas as pd
from pathlib import Path

df = pd.read_csv("../../data_pipeline/data/raw/enhanced_motor_vehicle_repair_towing_dataset.csv", low_memory=False)
print("Filas iniciales:", f"{len(df):,}")
print(df.isna().sum()[df.isna().sum()>0].to_string())
print("Duplicados:", int(df.duplicated().sum()))

<details><summary><b>Salida ejecutada</b></summary>

```text
Filas iniciales: 2,000,000
Towing Date 1714262 | Tow Location 1714262 | Drop-off Location 1714262 | Tow Truck Driver 1714262 | Customer Feedback 2000000
Duplicados: 0
```
</details>

## 1. Columnas inutilizables

In [ ]:
# Customer Feedback 100 % nula y Service Status corrupta
df = df.drop(columns=["Customer Feedback", "Service Status"])
print("Columnas restantes:", df.shape[1])

<details><summary><b>Salida ejecutada</b></summary>

```text
Columnas restantes: 26
```
</details>

## 2. Nulos estructurales MAR — grúa (RNF-03)

El nulo coincide 1:1 con `Service Type != 'Towing'`. Se marca 'No aplica' en texto; `Towing Date` no se imputa (nada inventado).

In [ ]:
mask = df["Service Type"] != "Towing"
for col in ["Tow Location","Drop-off Location","Tow Truck Driver"]:
    assert df[col].isna().sum() == mask.sum()
    df[col] = df[col].fillna("No aplica (servicio sin remolque)")
print("Columnas NOTOWING imputadas:", int(mask.sum()), "filas cada una")

<details><summary><b>Salida ejecutada</b></summary>

```text
Columnas NOTOWING imputadas: 1,714,262 filas cada una
```
</details>

## 3. IQR segmentado por tipo de servicio → reglas_iqr_servicio

In [ ]:
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return q1, q3, iqr, q1 - 1.5*iqr, q3 + 1.5*iqr

outliers = []
for col in ["Total Cost", "Estimated Cost"]:
    bandera = col.replace(" ","_") + "_Outlier_IQR"
    df[bandera] = False
    for tipo, sub in df.groupby("Service Type")[col]:
        q1, q3, iqr, lo, hi = iqr_bounds(sub)
        idx = sub[(sub < lo) | (sub > hi)].index
        df.loc[idx, bandera] = True
        outliers.append((col, tipo, lo, hi, len(idx)))
    print(col, "outliers:", int(df[bandera].sum()), f"({df[bandera].mean()*100:.3f}%)")

<details><summary><b>Salida ejecutada</b></summary>

```text
Total Cost: 11,830 outliers (0.592%)
Estimated Cost: 13,310 outliers (0.665%)
```
</details>

## 4. Validación y exportación

Produce `data_pipeline/data/processed/motor_vehicle_limpio_propio.csv` (2,000,000 × 31), `data_pipeline/outputs/bitacora_limpieza.csv` y `reglas_iqr_servicio.csv` (14 reglas para 7 tipos de servicio). Solo el IQR segmentado evita marcar reparaciones caras legítimas como anomalías.